In [1]:
import pandas as pd
import glob

USED_CSVS_GLOB = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/*_wg.csv"

OLD_PREFIX = "/home/al.pedro.santos/gaussian_football"
NEW_PREFIX = "/mnt/storage_C4/gaussian_football"

for csv_path in glob.glob(USED_CSVS_GLOB):
    df = pd.read_csv(csv_path)
    df["clip_path"] = df["clip_path"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    df["mel_path"] = df["mel_path"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    df.to_csv(csv_path, index=False)
    print(f"{csv_path}: {len(df)} linhas atualizadas")

/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/ligue-1_wg.csv: 12320 linhas atualizadas
/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/serie-a_wg.csv: 40128 linhas atualizadas
/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/bundesliga_wg.csv: 19732 linhas atualizadas
/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/epl_wg.csv: 34090 linhas atualizadas
/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/ucl_wg.csv: 35860 linhas atualizadas
/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/laliga_wg.csv: 43890 linhas atualizadas


In [2]:
import pandas as pd
import os
import glob

USED_CSVS_GLOB = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/*_wg.csv"

missing_clip = []
missing_mel = []
total_rows = 0

for csv_path in sorted(glob.glob(USED_CSVS_GLOB)):
    liga = os.path.basename(csv_path).replace(".csv", "")
    df = pd.read_csv(csv_path)
    total_rows += len(df)

    for clip_path in df["clip_path"]:
        if not os.path.exists(clip_path):
            missing_clip.append((liga, clip_path))

    if "mel_path" in df.columns:
        for mel_path in df["mel_path"].dropna():
            if not os.path.exists(mel_path):
                missing_mel.append((liga, mel_path))

print(f"Total de linhas verificadas: {total_rows}")
print(f"clip_path faltando: {len(missing_clip)}")
print(f"mel_path faltando: {len(missing_mel)}")

if missing_clip:
    with open("missing_clip_paths.txt", "w") as f:
        for liga, p in missing_clip:
            f.write(f"{liga}\t{p}\n")

if missing_mel:
    with open("missing_mel_paths.txt", "w") as f:
        for liga, p in missing_mel:
            f.write(f"{liga}\t{p}\n")

Total de linhas verificadas: 186020
clip_path faltando: 0
mel_path faltando: 0


### Juntar em um único csv

In [4]:
import pandas as pd
import os

# Caminho da pasta pai conforme a imagem
pasta_pai = '/mnt/storage_C4/gaussian_football/data/balanced_datasets/used'

# Nomes exatos dos 6 ficheiros
ficheiros = [
    'bundesliga_wg.csv', 
    'epl_wg.csv', 
    'laliga_wg.csv', 
    'ligue-1_wg.csv', 
    'serie-a_wg.csv', 
    'ucl_wg.csv'
]

# Lista para guardar cada DataFrame temporariamente
lista_de_dataframes = []

print("A iniciar a leitura dos ficheiros...")

# Ler cada ficheiro e adicionar à lista
for ficheiro in ficheiros:
    # Cria o caminho completo (pasta + nome do ficheiro)
    caminho_completo = os.path.join(pasta_pai, ficheiro)
    
    # Lê o CSV
    df = pd.read_csv(caminho_completo)
    lista_de_dataframes.append(df)
    print(f"Ficheiro lido: {ficheiro}")

# Juntar todos os ficheiros num único DataFrame
print("A juntar os ficheiros...")
df_combinado = pd.concat(lista_de_dataframes, ignore_index=True)

# Definir o nome do novo ficheiro final e guardá-lo na mesma pasta
caminho_final = os.path.join(pasta_pai, 'todas_as_ligas_combinadas_wg.csv')
df_combinado.to_csv(caminho_final, index=False)

print(f"Sucesso! O ficheiro final foi guardado em: {caminho_final}")

A iniciar a leitura dos ficheiros...
Ficheiro lido: bundesliga_wg.csv
Ficheiro lido: epl_wg.csv
Ficheiro lido: laliga_wg.csv
Ficheiro lido: ligue-1_wg.csv
Ficheiro lido: serie-a_wg.csv
Ficheiro lido: ucl_wg.csv
A juntar os ficheiros...
Sucesso! O ficheiro final foi guardado em: /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_combinadas_wg.csv


In [5]:
import pandas as pd
import os

# Caminho da pasta pai
pasta_pai = '/mnt/storage_C4/gaussian_football/data/balanced_datasets/used'
caminho_combinado = os.path.join(pasta_pai, 'todas_as_ligas_combinadas_wg.csv')

print("A ler o ficheiro combinado...")
df = pd.read_csv(caminho_combinado)

# Garantir que os valores da coluna estão em texto e minúsculas para evitar erros de correspondência
df['split'] = df['split'].astype(str).str.lower().str.strip()

# Mapeamento do valor da coluna 'split' para o nome exato do ficheiro pretendido
mapeamento_ficheiros = {
    'train': 'todas_as_ligas_train_wg.csv',
    'test': 'todas_as_ligas_test_wg.csv',
    'valid': 'todas_as_ligas_valid_wg.csv'
}

print("\nA iniciar a separação dos ficheiros...")

# Filtrar e guardar cada um dos ficheiros desejados
for valor_split, nome_ficheiro in mapeamento_ficheiros.items():
    # Filtra as linhas correspondentes
    df_filtrado = df[df['split'] == valor_split]
    
    # Se existirem dados para este split, guarda o ficheiro
    if not df_filtrado.empty:
        caminho_salvar = os.path.join(pasta_pai, nome_ficheiro)
        
        # Se for o caso de 'validation', ele vai guardar como 'todas_as_ligas_val.csv'.
        # Para evitar duplicar o print caso existam ambos, verificamos se o ficheiro já foi criado.
        if os.path.exists(caminho_salvar) and valor_split == 'validation':
            continue
            
        df_filtrado.to_csv(caminho_salvar, index=False)
        print(f"Sucesso! Criado: '{nome_ficheiro}' com {len(df_filtrado)} linhas.")
    else:
        # Só avisa se não encontrar 'train', 'test' ou 'val' (ignora se 'validation' não existir)
        if valor_split != 'validation':
            print(f"Aviso: Não foram encontradas linhas com o valor '{valor_split}' na coluna 'split'.")

print("\nProcesso concluído!")

A ler o ficheiro combinado...

A iniciar a separação dos ficheiros...
Sucesso! Criado: 'todas_as_ligas_train_wg.csv' com 112792 linhas.
Sucesso! Criado: 'todas_as_ligas_test_wg.csv' com 37394 linhas.
Sucesso! Criado: 'todas_as_ligas_valid_wg.csv' com 35834 linhas.

Processo concluído!
